# NLP Workflow - Perplexity Computation for N-gram models

## This Notebook Covers
- Modeling using N-grams on isolated randomized sample of Moderate Churn Risk Band (churn_intent in [0.3,0.8) )
- Building unigram and bigram vacabularies
- Modelling LDA for 5 topics using unigrams and bigrams
- Calculating and comparing perplexities

## Libraries Used
- **Python:** pandas, sklearn

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
from tqdm import tqdm
import time

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
df = pd.read_csv('data/complaints_with_nlp_features.csv',engine="python",
    on_bad_lines="skip")
df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,narrative_clean,narrative_no_stopwords,sentiment_compound,sentiment_neg,sentiment_neu,sentiment_pos,urgency_score,churn_intent,loyalty_score,text_length,word_count
0,01/20/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,I am writing to have the following information...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,92345,NaN,Consent provided,Web,01/20/25,Closed with non-monetary relief,Yes,NaN,11588109,i am writing to have the following information...,writing following information removed credit f...,0.9430,0.037,0.794,0.169,0.0,0.4,0.0,662,119
1,07/03/24,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,I am a victim of identity theft. Please delete...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,32824,NaN,Consent provided,Web,07/03/24,Closed with non-monetary relief,Yes,NaN,9416677,i am a victim of identity theft. please delete...,victim identity theft please delete remove ite...,0.2732,0.100,0.760,0.139,0.0,0.0,0.0,370,68
2,09/14/25,Vehicle loan or lease,Loan,Incorrect information on your report,Information belongs to someone else,"My name is XXXX XXXX, and I am formally disput...",NaN,"SANTANDER HOLDINGS USA, INC.",PA,19143,NaN,Consent provided,Web,09/14/25,Closed with explanation,Yes,NaN,15930829,"my name is xxxx xxxx, and i am formally disput...",name xxxx xxxx formally disputing fraudulent a...,-0.9042,0.151,0.766,0.083,1.0,0.4,0.0,830,140
3,05/01/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,"Upon reviewing my credit report, I have identi...",Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,76105,NaN,Consent provided,Web,05/01/25,Closed with non-monetary relief,Yes,NaN,13274568,"upon reviewing my credit report, i have identi...",upon reviewing credit report identified inaccu...,0.3818,0.000,0.867,0.133,0.0,0.0,0.0,127,18
4,12/08/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,Everything is explained in my resolution packa...,Company believes it acted appropriately as aut...,Kubota North America Corporation,MS,391XX,NaN,Consent provided,Web,12/08/25,Closed with explanation,Yes,NaN,17837761,everything is explained in my resolution packa...,everything explained resolution package ive al...,0.3818,0.000,0.860,0.140,0.0,0.0,0.0,107,17


In [5]:
df_text = df[df['narrative_no_stopwords'].notna()].copy()
    
print(f"\nDataset: {len(df_text):,} complaints with text")


Dataset: 1,400,010 complaints with text


In [6]:
# ─── Remove redacted terms from text ───
print("\nCleaning redacted CFPB terms (XXXX, XX, etc.)...")
import re 
def clean_redacted_terms(text):
        if not text or pd.isna(text):
            return ""
        
        # Remove all patterns with X's (case insensitive)
        text = re.sub(r'\b[xX]+\b', '', text)
        text = re.sub(r'\b[xX]{2,}', '', text)
        
        # Remove standalone numbers (often redacted dates/amounts)
        text = re.sub(r'\b\d+\b', '', text)
        
        # Remove very short tokens
        text = re.sub(r'\b\w{1,2}\b', '', text)
        
        # Clean whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
df_text['narrative_cleaned'] = df_text['narrative_no_stopwords'].apply(clean_redacted_terms)


Cleaning redacted CFPB terms (XXXX, XX, etc.)...


In [7]:
# Checking removed quantity
avg_len_before = df_text['narrative_no_stopwords'].str.len().mean()
avg_len_after = df_text['narrative_cleaned'].str.len().mean()
reduction_pct = (avg_len_before - avg_len_after) / avg_len_before * 100
    
print(f"   Avg text length before: {avg_len_before:.0f} chars")
print(f"   Avg text length after:  {avg_len_after:.0f} chars")
print(f"   Reduction: {reduction_pct:.1f}%")

   Avg text length before: 732 chars
   Avg text length after:  635 chars
   Reduction: 13.3%


In [8]:
df_text.head(10)

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,narrative_clean,narrative_no_stopwords,sentiment_compound,sentiment_neg,sentiment_neu,sentiment_pos,urgency_score,churn_intent,loyalty_score,text_length,word_count,narrative_cleaned
0,01/20/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,I am writing to have the following information...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,92345,NaN,Consent provided,Web,01/20/25,Closed with non-monetary relief,Yes,NaN,11588109,i am writing to have the following information...,writing following information removed credit f...,0.9430,0.037,0.794,0.169,0.0,0.4,0.0,662,119,writing following information removed credit f...
1,07/03/24,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,I am a victim of identity theft. Please delete...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,32824,NaN,Consent provided,Web,07/03/24,Closed with non-monetary relief,Yes,NaN,9416677,i am a victim of identity theft. please delete...,victim identity theft please delete remove ite...,0.2732,0.100,0.760,0.139,0.0,0.0,0.0,370,68,victim identity theft please delete remove ite...
2,09/14/25,Vehicle loan or lease,Loan,Incorrect information on your report,Information belongs to someone else,"My name is XXXX XXXX, and I am formally disput...",NaN,"SANTANDER HOLDINGS USA, INC.",PA,19143,NaN,Consent provided,Web,09/14/25,Closed with explanation,Yes,NaN,15930829,"my name is xxxx xxxx, and i am formally disput...",name xxxx xxxx formally disputing fraudulent a...,-0.9042,0.151,0.766,0.083,1.0,0.4,0.0,830,140,name formally disputing fraudulent auto loan s...
3,05/01/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,"Upon reviewing my credit report, I have identi...",Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,76105,NaN,Consent provided,Web,05/01/25,Closed with non-monetary relief,Yes,NaN,13274568,"upon reviewing my credit report, i have identi...",upon reviewing credit report identified inaccu...,0.3818,0.000,0.867,0.133,0.0,0.0,0.0,127,18,upon reviewing credit report identified inaccu...
4,12/08/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,Everything is explained in my resolution packa...,Company believes it acted appropriately as aut...,Kubota North America Corporation,MS,391XX,NaN,Consent provided,Web,12/08/25,Closed with explanation,Yes,NaN,17837761,everything is explained in my resolution packa...,everything explained resolution package ive al...,0.3818,0.000,0.860,0.140,0.0,0.0,0.0,107,17,everything explained resolution package ive al...
5,03/06/24,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Investigation took more than 30 days,I have already sent a letter addressing the in...,NaN,"EQUIFAX, INC.",CA,90806,NaN,Consent provided,Web,03/06/24,Closed with non-monetary relief,Yes,NaN,8488899,i have already sent a letter addressing the in...,already sent letter addressing inaccuracies un...,-0.3182,0.116,0.778,0.107,1.0,0.0,0.0,725,127,already sent letter addressing inaccuracies un...
6,05/08/25,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Was not notified of investigation status or re...,I reviewed my credit report and noticed severa...,NaN,"EQUIFAX, INC.",MA,021

In [12]:
df = df_text.copy(deep=True)
df.head(2)

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,narrative_clean,narrative_no_stopwords,sentiment_compound,sentiment_neg,sentiment_neu,sentiment_pos,urgency_score,churn_intent,loyalty_score,text_length,word_count,narrative_cleaned
0,01/20/25,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,I am writing to have the following information...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",CA,92345,NaN,Consent provided,Web,01/20/25,Closed with non-monetary relief,Yes,NaN,11588109,i am writing to have the following information...,writing following information removed credit f...,0.9430,0.037,0.794,0.169,0.0,0.4,0.0,662,119,writing following information removed credit f...
1,07/03/24,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,I am a victim of identity theft. Please delete...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,32824,NaN,Consent provided,Web,07/03/24,Closed with non-monetary relief,Yes,NaN,9416677,i am a victim of identity theft. please delete...,victim identity theft please delete remove ite...,0.2732,0.100,0.760,0.139,0.0,0.0,0.0,370,68,victim identity theft please delete remove ite...


In [14]:
df.to_csv('data/complaints_with_nlp_features_deep_clean.csv', index=False)

In [15]:
# Check text lengths
df['text_length'] = df['narrative_cleaned'].str.len()
print(f"\nText length statistics:")
print(f"   Mean:   {df['text_length'].mean():.0f} characters")
print(f"   Median: {df['text_length'].median():.0f} characters")
print(f"   Min:    {df['text_length'].min():.0f} characters")
print(f"   Max:    {df['text_length'].max():.0f} characters")


Text length statistics:
   Mean:   635 characters
   Median: 440 characters
   Min:    0 characters
   Max:    31615 characters


In [16]:
print("Criteria: churn_intent between 0.3 and 0.8")

moderate_risk = df[
    (df['churn_intent'] >= 0.3) & 
    (df['churn_intent'] < 0.8)
].copy()

Criteria: churn_intent between 0.3 and 0.8


In [17]:
print(f"\nModerate risk complaints: {len(moderate_risk):,}")
print(f"Percentage of total: {len(moderate_risk)/len(df)*100:.1f}%")


Moderate risk complaints: 202,545
Percentage of total: 14.5%


In [18]:
SAMPLE_SIZE = 30000

if len(moderate_risk) > SAMPLE_SIZE:
    sample_df = moderate_risk.sample(n=SAMPLE_SIZE, random_state=42)
    print(f"   Sampled {SAMPLE_SIZE:,} complaints (random seed=42)")
else:
    sample_df = moderate_risk.copy()
    print(f"   Using all {len(sample_df):,} complaints (less than target)")

   Sampled 30,000 complaints (random seed=42)


In [19]:
print(sample_df.shape)
sample_df.head(5)

(30000, 30)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,narrative_clean,narrative_no_stopwords,sentiment_compound,sentiment_neg,sentiment_neu,sentiment_pos,urgency_score,churn_intent,loyalty_score,text_length,word_count,narrative_cleaned
1121521,09/12/25,Mortgage,Conventional home mortgage,Applying for a mortgage or refinancing an exis...,Delays in the application process,"Originally in XXXX of XXXX, I requested PMI ca...",Company believes it acted appropriately as aut...,"Shellpoint Partners, LLC",MD,21702,NaN,Consent provided,Web,09/12/25,Closed with explanation,Yes,NaN,15914989,"originally in xxxx of xxxx, i requested pmi ca...",originally xxxx xxxx requested pmi cancellatio...,-0.7636,0.063,0.894,0.043,0.7,0.4,0.2,1084,358,originally requested pmi cancellation believed...
827346,01/15/25,"Money transfer, virtual currency, or money ser...",Domestic (US) money transfer,Other transaction problem,NaN,I am writing to formally raise my concerns abo...,NaN,"Early Warning Services, LLC",IL,60619,NaN,Consent provided,Web,01/15/25,Closed with explanation,Yes,NaN,11551991,i am writing to formally raise my concerns abo...,writing formally raise concerns zelles handlin...,-0.9809,0.226,0.740,0.034,1.0,0.7,0.0,718,149,writing formally raise concerns zelles handlin...
1258140,08/07/25,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,I reviewed my credit report and discovered sev...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,AL,36301,NaN,Consent provided,Web,08/07/25,Closed with explanation,Yes,NaN,15142157,i reviewed my credit report and discovered sev...,reviewed credit report discovered several inac...,-0.8202,0.122,0.801,0.077,0.0,0.4,0.0,722,166,reviewed credit report discovered several inac...
1351712,07/08/25,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Their investigation did not fix an error on yo...,XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX X...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,OH,44312,NaN,Consent provided,Web,07/08/25,Closed with explanation,Yes,NaN,14513733,xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx x...,xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx xxxx x...,0.1811,0.090,0.798,0.112,1.0,0.4,0.2,1041,262,//year experian subject urgent request immedia...
1358497,08/09/23,"Money transfer, virtual currency, or money ser...",Mobile or digital wallet,Fraud or scam,NaN,Few Days back 2 People XXXX XXXX ( XXXX : XXXX...,Company has responded to the consumer and the ...,Payward Ventures Inc. dba Kraken,PA,19468,NaN,Consent provided,Web,08/09/23,Closed with explanation,Yes,NaN,7372749,few days back 2 people xxxx xxxx ( xxxx : xxxx...,days back 2 people xxxx xxxx xxxx xxxx xxxx xx...,0.9963,0.042,0.840,0.118,1.0,0.4,0.0,2749,829,days back people assistant approached saying g...


In [20]:
print(f"\nSample characteristics:")
print(f"Sentiment mean:     {sample_df['sentiment_compound'].mean():.3f}")
print(f"Churn intent mean:  {sample_df['churn_intent'].mean():.3f}")
print(f"Urgency mean:       {sample_df['urgency_score'].mean():.3f}")


Sample characteristics:
Sentiment mean:     -0.150
Churn intent mean:  0.452
Urgency mean:       0.637


In [21]:
from sklearn.model_selection import train_test_split

# Split data for perplexity evaluation
# Train set: build vocabulary and train LDA
# Test set: calculate perplexity (unseen data)

print("\nSplitting data into train/test...")
print("Split ratio: 80% train, 20% test")
print("Random seed: 42")

train_texts, test_texts = train_test_split(
    sample_df['narrative_cleaned'],
    test_size=0.2,
    random_state=42
)

print(f"\nTraining set:   {len(train_texts):,} complaints")
print(f"Test set:       {len(test_texts):,} complaints")

# Verify split
total = len(train_texts) + len(test_texts)
print(f"Total:          {total:,} complaints")
print(f"Train %:        {len(train_texts)/total*100:.1f}%")
print(f"Test %:         {len(test_texts)/total*100:.1f}%")

# Save indices for later inspection
train_indices = train_texts.index.tolist()
test_indices = test_texts.index.tolist()

print(f"\nSplit complete")


Splitting data into train/test...
Split ratio: 80% train, 20% test
Random seed: 42

Training set:   24,000 complaints
Test set:       6,000 complaints
Total:          30,000 complaints
Train %:        80.0%
Test %:         20.0%

Split complete


## Unigram Vocabulary

In [22]:
from sklearn.feature_extraction.text import CountVectorizer
import time

# Starting with unigrams only (simpler, faster)

print("\nCreating CountVectorizer (unigrams only)...")

vectorizer_unigram = CountVectorizer(
    max_features=5000,       # Limit vocabulary size
    min_df=5,                # Word must appear in at least 5 documents
    max_df=0.7,              # Remove words in >70% of documents
    ngram_range=(1, 1),      # Only unigrams
    token_pattern=r'(?u)\b\w+\b'
)

print("Parameters:")
print(f"max_features = 5000 (top 5000 words by frequency)")
print(f"min_df = 5 (minimum document frequency)")
print(f"max_df = 0.7 (maximum document frequency)")
print(f"ngram_range = (1,1) (unigrams only)")


Creating CountVectorizer (unigrams only)...
Parameters:
max_features = 5000 (top 5000 words by frequency)
min_df = 5 (minimum document frequency)
max_df = 0.7 (maximum document frequency)
ngram_range = (1,1) (unigrams only)


In [23]:
# Fit on training data
print(f"\nFitting vectorizer on {len(train_texts):,} training texts...")

start_time = time.time()
train_bow_unigram = vectorizer_unigram.fit_transform(train_texts)
fit_time = time.time() - start_time

print(f"Completed in {fit_time:.2f} seconds")


Fitting vectorizer on 24,000 training texts...
Completed in 1.91 seconds


In [24]:
vocab = vectorizer_unigram.get_feature_names_out()
print(f"\nVocabulary statistics:")
print(f"Total words: {len(vocab):,}")
print(f"First 20 words: {list(vocab[:20])}")
print(f"Last 20 words: {list(vocab[-20:])}")

print(f"\nDocument-Term Matrix:")
print(f"Shape: {train_bow_unigram.shape}")
print(f"(rows = documents, columns = vocabulary words)")
print(f"Total elements: {train_bow_unigram.shape[0] * train_bow_unigram.shape[1]:,}")
print(f"Non-zero elements: {train_bow_unigram.nnz:,}")


Vocabulary statistics:
Total words: 5,000
First 20 words: ['1028a', '1099c', '1232g', '1581a', '15u', '15usc', '1666b', '1681a', '1681b', '1681c', '1681c2', '1681e', '1681g', '1681h', '1681i', '1681j', '1681m', '1681n', '1681o', '1681q']
Last 20 words: ['wrongly', 'wrote', 'yall', 'year', 'yearly', 'years', 'yes', 'yesterday', 'yet', 'yield', 'york', 'you', 'young', 'youre', 'youve', 'yrs', 'zelle', 'zelles', 'zero', 'zip']

Document-Term Matrix:
Shape: (24000, 5000)
(rows = documents, columns = vocabulary words)
Total elements: 120,000,000
Non-zero elements: 1,995,219


In [25]:
sparsity = 1.0 - (train_bow_unigram.nnz / (train_bow_unigram.shape[0] * train_bow_unigram.shape[1]))
print(f"   Sparsity: {sparsity*100:.2f}%")
print(f"   (Most entries are zero - text data is sparse!)")

# Transform test data
print(f"\nTransforming test data...")
test_bow_unigram = vectorizer_unigram.transform(test_texts)
print(f"Test matrix shape: {test_bow_unigram.shape}")

   Sparsity: 98.34%
   (Most entries are zero - text data is sparse!)

Transforming test data...
Test matrix shape: (6000, 5000)


In [26]:
from sklearn.decomposition import LatentDirichletAllocation
import time

lda_unigram = LatentDirichletAllocation(
    n_components=5,              # 5 topics
    max_iter=20,                 # Maximum iterations
    learning_method='online',    # Faster than 'batch'
    learning_offset=50.,         # Helps with convergence
    random_state=42,             # Reproducibility
    n_jobs=-1,                   # Use all CPU cores
    verbose=1                    # Show progress
)

print("Parameters:")
print(f"n_components = 5 (number of topics)")
print(f"max_iter = 20 (maximum iterations)")
print(f"learning_method = 'online' (incremental learning)")
print(f"random_state = 42 (reproducible results)")

# Train the model
print(f"\nTraining LDA on {train_bow_unigram.shape[0]:,} documents...")

start_train = time.time()
lda_unigram.fit(train_bow_unigram)
train_time = time.time() - start_train

print(f"\nTraining completed in {train_time:.1f} seconds ({train_time/60:.1f} minutes)")

# Check training perplexity
train_perplexity = lda_unigram.perplexity(train_bow_unigram)
print(f"\nTraining perplexity: {train_perplexity:.2f}")
# print(f"(Lower is better - measures how well model fits training data)")


Parameters:
n_components = 5 (number of topics)
max_iter = 20 (maximum iterations)
learning_method = 'online' (incremental learning)
random_state = 42 (reproducible results)

Training LDA on 24,000 documents...
iteration: 1 of max_iter: 20
iteration: 2 of max_iter: 20
iteration: 3 of max_iter: 20
iteration: 4 of max_iter: 20
iteration: 5 of max_iter: 20
iteration: 6 of max_iter: 20
iteration: 7 of max_iter: 20
iteration: 8 of max_iter: 20
iteration: 9 of max_iter: 20
iteration: 10 of max_iter: 20
iteration: 11 of max_iter: 20
iteration: 12 of max_iter: 20
iteration: 13 of max_iter: 20
iteration: 14 of max_iter: 20
iteration: 15 of max_iter: 20
iteration: 16 of max_iter: 20
iteration: 17 of max_iter: 20
iteration: 18 of max_iter: 20
iteration: 19 of max_iter: 20
iteration: 20 of max_iter: 20

Training completed in 106.8 seconds (1.8 minutes)

Training perplexity: 712.81


In [27]:
print("\n" + "=" * 70)
print("TEST PERPLEXITY (UNIGRAM)")
print("=" * 70)

# Calculate perplexity on unseen test data
# This measures how well the model generalizes

print(f"\nCalculating perplexity on test set...")
print(f"Test set size: {test_bow_unigram.shape[0]:,} documents")

test_perplexity_unigram = lda_unigram.perplexity(test_bow_unigram)

print(f"\nTest perplexity: {test_perplexity_unigram:.2f}")

# Compare train vs test
print(f"\nTrain vs Test Comparison:")
print(f"   Train perplexity: {train_perplexity:.2f}")
print(f"   Test perplexity:  {test_perplexity_unigram:.2f}")

difference = test_perplexity_unigram - train_perplexity
pct_increase = (difference / train_perplexity) * 100

print(f"   Difference:       {difference:.2f} ({pct_increase:+.1f}%)")

if pct_increase < 10:
    print(f"   Good generalization (< 10% increase)")
elif pct_increase < 20:
    print(f"   Moderate overfitting (10-20% increase)")
else:
    print(f"   High overfitting (> 20% increase)")


TEST PERPLEXITY (UNIGRAM)

Calculating perplexity on test set...
Test set size: 6,000 documents

Test perplexity: 742.34

Train vs Test Comparison:
   Train perplexity: 712.81
   Test perplexity:  742.34
   Difference:       29.53 (+4.1%)
   Good generalization (< 10% increase)


In [28]:
print("\nTop 10 words per topic:\n")

feature_names = vectorizer_unigram.get_feature_names_out()
n_top_words = 10

for topic_idx, topic in enumerate(lda_unigram.components_):
    top_indices = topic.argsort()[-n_top_words:][::-1]
    top_words = [feature_names[i] for i in top_indices]
    
    print(f"   Topic {topic_idx+1}: {', '.join(top_words)}")

print(f"\nTop 5 words with weights (Topic 1 example):\n")

topic_0 = lda_unigram.components_[0]
top_5_indices = topic_0.argsort()[-5:][::-1]

for idx in top_5_indices:
    word = feature_names[idx]
    weight = topic_0[idx]
    print(f"   {word:15s}: {weight:.4f}")

print(f"\nInterpretation:")
print(f"   Higher weight = word is more important to that topic")


Top 10 words per topic:

   Topic 1: credit, fcra, reporting, information, debt, consumer, report, act, dispute, request
   Topic 2: information, consumer, report, credit, reporting, identity, theft, section, agency, items
   Topic 3: account, credit, amount, report, date, reporting, late, payment, balance, information
   Topic 4: account, amount, bank, card, would, credit, told, received, loan, payment
   Topic 5: financial, unfair, cash, resolution, app, act, zelle, significant, failed, leaving

Top 5 words with weights (Topic 1 example):

   credit         : 21967.9257
   fcra           : 13539.9597
   reporting      : 12530.7813
   information    : 11582.4027
   debt           : 9617.8759

Interpretation:
   Higher weight = word is more important to that topic


In [29]:
import pickle
import json
from datetime import datetime
import os

with open('models/lda_unigram_model.pkl', 'wb') as f:
    pickle.dump(lda_unigram, f)

with open('models/vectorizer_unigram.pkl', 'wb') as f:
    pickle.dump(vectorizer_unigram, f)

unigram_results = {
    'model_type': 'unigram',
    'n_topics': 5,
    'vocabulary_size': len(feature_names),
    'train_size': len(train_texts),
    'test_size': len(test_texts),
    'training_time_seconds': round(train_time, 2),
    'train_perplexity': round(train_perplexity, 2),
    'test_perplexity': round(test_perplexity_unigram, 2),
    'overfitting_percent': round(pct_increase, 2),
    'max_iterations': 20,
    'timestamp': datetime.now().isoformat()
}

with open('outputs/perplexity_unigram_results.json', 'w') as f:
    json.dump(unigram_results, f, indent=2)

# Display the saved results
print(json.dumps(unigram_results, indent=2))

print("\n" + "=" * 70)
print("UNIGRAM MODEL SUMMARY")
print("=" * 70)
print(f"Vocabulary:       {len(feature_names):,} words")
print(f"Training time:    {train_time:.1f} seconds")
print(f"Train perplexity: {train_perplexity:.2f}")
print(f"Test perplexity:  {test_perplexity_unigram:.2f}")
print(f"Generalization:   {pct_increase:+.1f}% change")
print("=" * 70)

{
  "model_type": "unigram",
  "n_topics": 5,
  "vocabulary_size": 5000,
  "train_size": 24000,
  "test_size": 6000,
  "training_time_seconds": 106.75,
  "train_perplexity": 712.81,
  "test_perplexity": 742.34,
  "overfitting_percent": 4.14,
  "max_iterations": 20,
  "timestamp": "2026-03-03T22:18:52.642238"
}

UNIGRAM MODEL SUMMARY
Vocabulary:       5,000 words
Training time:    106.8 seconds
Train perplexity: 712.81
Test perplexity:  742.34
Generalization:   +4.1% change


## Bigram Vocabulary

In [30]:
from sklearn.feature_extraction.text import CountVectorizer
import time

vectorizer_bigram = CountVectorizer(
    max_features=8000,           # 
    min_df=10,                   # 
    max_df=0.7,
    ngram_range=(1, 2),          # Unigrams + Bigrams
    token_pattern=r'(?u)\b\w+\b'
)

print("\nParameters:")
print(f"max_features = 8,000 (top 8000 terms)")
print(f"min_df = 10 (must appear in ≥10 documents)")
print(f"max_df = 0.7")
print(f"ngram_range = (1,2) (unigrams + bigrams)")


Parameters:
max_features = 8,000 (top 8000 terms)
min_df = 10 (must appear in ≥10 documents)
max_df = 0.7
ngram_range = (1,2) (unigrams + bigrams)


In [31]:
# Fit on training data
print(f"\nBuilding bigram vocabulary from {len(train_texts):,} documents...")

start_vocab = time.time()
train_bow_bigram = vectorizer_bigram.fit_transform(train_texts)
vocab_time = time.time() - start_vocab

print(f"Completed in {vocab_time:.1f} seconds")


Building bigram vocabulary from 24,000 documents...
Completed in 7.8 seconds


In [32]:
# Inspect vocabulary
feature_names_bigram = vectorizer_bigram.get_feature_names_out()

# Count unigrams vs bigrams
unigrams_list = [f for f in feature_names_bigram if ' ' not in f]
bigrams_list = [f for f in feature_names_bigram if ' ' in f]

print(f"\nVocabulary composition:")
print(f"   Total terms:  {len(feature_names_bigram):,}")
print(f"   Unigrams:     {len(unigrams_list):,} ({len(unigrams_list)/len(feature_names_bigram)*100:.1f}%)")
print(f"   Bigrams:      {len(bigrams_list):,} ({len(bigrams_list)/len(feature_names_bigram)*100:.1f}%)")

print(f"\n   Example bigrams: {bigrams_list[:10]}")


Vocabulary composition:
   Total terms:  8,000
   Unigrams:     3,137 (39.2%)
   Bigrams:      4,863 (60.8%)

   Example bigrams: ['1028a aggravated', '1666b creditor', '1666b time', '1666b timing', '1681a definitions', '1681a fair', '1681a section', '1681b consumer', '1681b established', '1681b outlines']


In [33]:
# Matrix stats
print(f"\nDocument-Term Matrix:")
print(f"Shape: {train_bow_bigram.shape}")
print(f"Non-zero: {train_bow_bigram.nnz:,}")

sparsity_bigram = 1.0 - (train_bow_bigram.nnz / (train_bow_bigram.shape[0] * train_bow_bigram.shape[1]))
print(f"Sparsity: {sparsity_bigram*100:.2f}%")

# Transform test set
print(f"\nTransforming test set...")
test_bow_bigram = vectorizer_bigram.transform(test_texts)
print(f"Test shape: {test_bow_bigram.shape}")


Document-Term Matrix:
Shape: (24000, 8000)
Non-zero: 3,063,050
Sparsity: 98.40%

Transforming test set...
Test shape: (6000, 8000)


In [34]:
from sklearn.decomposition import LatentDirichletAllocation
import time

lda_bigram = LatentDirichletAllocation(
    n_components=5,
    max_iter=20,
    learning_method='online',
    learning_offset=50.,
    batch_size=128,      
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Parameters:")
print(f"n_components = 5")
print(f"max_iter = 20")
print(f"batch_size = 128")

# Train
print(f"\nTraining bigram LDA...")
print(f"Documents: {train_bow_bigram.shape[0]:,}")
print(f"Features:  {train_bow_bigram.shape[1]:,}")

start_train = time.time()
lda_bigram.fit(train_bow_bigram)
train_time_bigram = time.time() - start_train

print(f"\nTraining completed in {train_time_bigram:.1f} seconds ({train_time_bigram/60:.1f} minutes)")

# Training perplexity
train_perplexity_bigram = lda_bigram.perplexity(train_bow_bigram)
print(f"\nTraining perplexity: {train_perplexity_bigram:.2f}")

Parameters:
n_components = 5
max_iter = 20
batch_size = 128

Training bigram LDA...
Documents: 24,000
Features:  8,000
iteration: 1 of max_iter: 20
iteration: 2 of max_iter: 20
iteration: 3 of max_iter: 20
iteration: 4 of max_iter: 20
iteration: 5 of max_iter: 20
iteration: 6 of max_iter: 20
iteration: 7 of max_iter: 20
iteration: 8 of max_iter: 20
iteration: 9 of max_iter: 20
iteration: 10 of max_iter: 20
iteration: 11 of max_iter: 20
iteration: 12 of max_iter: 20
iteration: 13 of max_iter: 20
iteration: 14 of max_iter: 20
iteration: 15 of max_iter: 20
iteration: 16 of max_iter: 20
iteration: 17 of max_iter: 20
iteration: 18 of max_iter: 20
iteration: 19 of max_iter: 20
iteration: 20 of max_iter: 20

Training completed in 107.1 seconds (1.8 minutes)

Training perplexity: 1275.61


In [35]:
print("\n" + "=" * 70)
print("TEST PERPLEXITY (BIGRAM)")
print("=" * 70)

# Calculate test perplexity
print(f"\nCalculating test perplexity...")

test_perplexity_bigram = lda_bigram.perplexity(test_bow_bigram)

print(f"Test perplexity: {test_perplexity_bigram:.2f}")

# Compare train vs test (bigram)
print(f"\nBigram Train vs Test:")
print(f"   Train: {train_perplexity_bigram:.2f}")
print(f"   Test:  {test_perplexity_bigram:.2f}")

diff_bigram = test_perplexity_bigram - train_perplexity_bigram
pct_bigram = (diff_bigram / train_perplexity_bigram) * 100

print(f"   Difference: {diff_bigram:.2f} ({pct_bigram:+.1f}%)")

# Compare UNIGRAM vs BIGRAM on test set
print(f"\nUNIGRAM vs BIGRAM Comparison (TEST SET):")
print(f"   Unigram test perplexity: {test_perplexity_unigram:.2f}")
print(f"   Bigram test perplexity:  {test_perplexity_bigram:.2f}")

improvement = test_perplexity_unigram - test_perplexity_bigram
improvement_pct = (improvement / test_perplexity_unigram) * 100

print(f"\nImprovement: {improvement:.2f} points ({improvement_pct:+.1f}%)")

if improvement > 0:
    print(f"Bigram is BETTER (lower perplexity)")
    print(f"Bigrams capture useful phrase patterns")
else:
    print(f"Bigram is WORSE (higher perplexity)")
    print(f"Bigrams may add noise, stick with unigrams")


TEST PERPLEXITY (BIGRAM)

Calculating test perplexity...
Test perplexity: 1360.56

Bigram Train vs Test:
   Train: 1275.61
   Test:  1360.56
   Difference: 84.95 (+6.7%)

UNIGRAM vs BIGRAM Comparison (TEST SET):
   Unigram test perplexity: 742.34
   Bigram test perplexity:  1360.56

Improvement: -618.22 points (-83.3%)
Bigram is WORSE (higher perplexity)
Bigrams may add noise, stick with unigrams


In [36]:
print("\nTop 15 terms per topic (bigram model):\n")

n_top_terms = 15

for topic_idx, topic in enumerate(lda_bigram.components_):
    top_indices = topic.argsort()[-n_top_terms:][::-1]
    top_terms = [feature_names_bigram[i] for i in top_indices]
    
    top_unigrams = [t for t in top_terms if ' ' not in t]
    top_bigrams = [t for t in top_terms if ' ' in t]
    
    print(f"Topic {topic_idx+1}:")
    print(f"Unigrams: {', '.join(top_unigrams[:10])}")
    if top_bigrams:
        print(f"Bigrams:  {', '.join(top_bigrams[:5])}")
    print()


Top 15 terms per topic (bigram model):

Topic 1:
Unigrams: credit, account, reporting, report, information, fcra, inaccurate, amount, debt, accounts
Bigrams:  credit report, credit reporting

Topic 2:
Unigrams: report, credit, identity, theft, information, items, accounts, consumer, fraudulent, please
Bigrams:  identity theft, credit report

Topic 3:
Unigrams: account, amount, credit, bank, card, would, told, payment, received, closed

Topic 4:
Unigrams: consumer, information, reporting, section, credit, agency, report, states, usc, financial
Bigrams:  consumer reporting, reporting agency, consumer report

Topic 5:
Unigrams: cash, financial, app, unfair, resolution, act, practices, protection, deceptive, significant
Bigrams:  cash app, unfair deceptive, financial protection



In [37]:
import pickle
import json
from datetime import datetime
import pandas as pd

with open('models/lda_bigram_model.pkl', 'wb') as f:
    pickle.dump(lda_bigram, f)

with open('models/vectorizer_bigram.pkl', 'wb') as f:
    pickle.dump(vectorizer_bigram, f)

# Save bigram results
bigram_results = {
    'model_type': 'bigram',
    'n_topics': 5,
    'vocabulary_size': len(feature_names_bigram),
    'n_unigrams': len(unigrams_list),
    'n_bigrams': len(bigrams_list),
    'train_size': len(train_texts),
    'test_size': len(test_texts),
    'vocab_build_time_seconds': round(vocab_time, 2),
    'training_time_seconds': round(train_time_bigram, 2),
    'train_perplexity': round(train_perplexity_bigram, 2),
    'test_perplexity': round(test_perplexity_bigram, 2),
    'overfitting_percent': round(pct_bigram, 2),
    'timestamp': datetime.now().isoformat()
}

with open('outputs/perplexity_bigram_results.json', 'w') as f:
    json.dump(bigram_results, f, indent=2)

comparison_data = {
    'Model': ['Unigram', 'Bigram'],
    'Vocabulary Size': [
        unigram_results['vocabulary_size'],
        bigram_results['vocabulary_size']
    ],
    'Train Time (s)': [
        unigram_results['training_time_seconds'],
        bigram_results['training_time_seconds']
    ],
    'Train Perplexity': [
        unigram_results['train_perplexity'],
        bigram_results['train_perplexity']
    ],
    'Test Perplexity': [
        unigram_results['test_perplexity'],
        bigram_results['test_perplexity']
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "=" * 70)
print("FINAL COMPARISON: UNIGRAM vs BIGRAM")
print("=" * 70)
print(comparison_df.to_string(index=False))
print("=" * 70)


FINAL COMPARISON: UNIGRAM vs BIGRAM
  Model  Vocabulary Size  Train Time (s)  Train Perplexity  Test Perplexity
Unigram             5000          106.75            712.81           742.34
 Bigram             8000          107.11           1275.61          1360.56


In [38]:
if test_perplexity_bigram < test_perplexity_unigram:
    winner = 'Bigram'
    improvement_val = test_perplexity_unigram - test_perplexity_bigram
    print(f"\nBetter Model here: {winner}")
    print(f"   Improvement: {improvement_val:.2f} points ({improvement_pct:.1f}% better)")
    print(f"   Recommendation: Use bigram model for better topic quality")
else:
    winner = 'Unigram'
    degradation = test_perplexity_bigram - test_perplexity_unigram
    print(f"\nBetter Model here: {winner}")
    print(f"   Bigrams increased perplexity by {degradation:.2f} points")
    print(f"   Recommendation: Use unigram model (simpler and better)")

comparison_summary = {
    'comparison_table': comparison_df.to_dict(orient='records'),
    'winner': winner,
    'test_perplexity_improvement': round(improvement, 2),
    'improvement_percent': round(improvement_pct, 2),
    'recommendation': f"Use {winner} model"
}

with open('outputs/perplexity_comparison.json', 'w') as f:
    json.dump(comparison_summary, f, indent=2)


Better Model here: Unigram
   Bigrams increased perplexity by 618.22 points
   Recommendation: Use unigram model (simpler and better)
